In [1]:
import sqlite3
import pandas as pd


from sqlalchemy import create_engine, event
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_groq import ChatGroq
import os

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
df= pd.read_csv("location/data", low_memory=True)
df.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.0,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.0,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.0,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.0,0.00,0,0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.0,0.00,0,0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.0,0.00,0,0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.00,0,0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.0,0.00,0,0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.0,40348.79,0,0


In [ ]:
# create/connect DB (this automatically creates fraud.db)
conn = sqlite3.connect("fraud.db")

# push dataframe to DB
df.to_sql("fraud_data", conn, if_exists="replace", index=False)

# verify
print(pd.read_sql("SELECT * FROM fraud_data LIMIT 5", conn))

conn.close()

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  


In [ ]:


def make_readonly_engine(db_path: str):
    """
    sqlite3 URI mode with mode=ro — OS-level read-only.
    Any DELETE/DROP/INSERT will raise OperationalError before executing.
    """
    def creator():
        return sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)

    engine = create_engine("sqlite:///", creator=creator)
    return engine

engine = make_readonly_engine("fraud.db")

In [7]:
# ── SQLDatabase using the read-only engine ────────────────────────────────────
db = SQLDatabase(
    engine,
    include_tables=["fraud_data"],
    sample_rows_in_table_info=2
)

In [5]:
# ── LLM ───────────────────────────────────────────────────────────────────────
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-120b",  # change to your target model
    temperature=0,
    max_tokens=1024,
)

In [8]:

# ── Agent ─────────────────────────────────────────────────────────────────────
SYSTEM = """You are a read-only fraud data analyst.
ONLY generate SELECT statements.
Never generate INSERT, UPDATE, DELETE, DROP, CREATE, ALTER, TRUNCATE.
If asked to modify data, say: 'I can only query data, not modify it.'

Table: fraud_data
Columns: step, type, amount, nameOrig, oldbalanceOrg, newbalanceOrig,
         nameDest, oldbalanceDest, newbalanceDest, isFraud, isFlaggedFraud
"""

agent = create_sql_agent(
    llm=llm,
    db=db,
    agent_type="openai-tools",
    verbose=True,
    prefix=SYSTEM
)


In [9]:
response = agent.invoke({"input": "how many fraud transactions are there?"})
print(response["output"])



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


fraud_data
Invoking: `sql_db_schema` with `{'table_names': 'fraud_data'}`



CREATE TABLE fraud_data (
	step INTEGER, 
	type TEXT, 
	amount REAL, 
	"nameOrig" TEXT, 
	"oldbalanceOrg" REAL, 
	"newbalanceOrig" REAL, 
	"nameDest" TEXT, 
	"oldbalanceDest" REAL, 
	"newbalanceDest" REAL, 
	"isFraud" INTEGER, 
	"isFlaggedFraud" INTEGER
)

/*
2 rows from fraud_data table:
step	type	amount	nameOrig	oldbalanceOrg	newbalanceOrig	nameDest	oldbalanceDest	newbalanceDest	isFraud	isFlaggedFraud
1	PAYMENT	9839.64	C1231006815	170136.0	160296.36	M1979787155	0.0	0.0	0	0
1	PAYMENT	1864.28	C1666544295	21249.0	19384.72	M2044282225	0.0	0.0	0	0
*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT COUNT(*) AS fraud_count FROM fraud_data WHERE isFraud = 1;'}`


SELECT COUNT(*) AS fraud_count FROM fraud_data WHERE isFraud = 1;
Invoking: `sql_db_query` with `{'query': 'SELECT COUNT(*) AS fraud_count FROM